# House Price Prediction — Regression & Regularization

Academic practical exploring regression models and regularization methods for predicting the `PRICE` variable.

In [ ]:
# import 'Pandas' 
import pandas as pd 

# import 'Numpy' 
import numpy as np

# import subpackage of Matplotlib
import matplotlib.pyplot as plt

# import 'Seaborn' 
import seaborn as sns

# to suppress warnings 
from warnings import filterwarnings
filterwarnings('ignore')

# import train-test split 
from sklearn.model_selection import train_test_split

# import function to perform linear regression using OLS
import statsmodels.api as sm

# 'metrics' from sklearn is used for evaluating the model performance
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

# import function to perform linear regression
from sklearn.linear_model import LinearRegression


# import StandardScaler to perform scaling
from sklearn.preprocessing import StandardScaler 

# import SGDRegressor from sklearn to perform linear regression with stochastic gradient descent
from sklearn.linear_model import SGDRegressor

# import function for ridge regression
from sklearn.linear_model import Ridge

# import function for lasso regression
from sklearn.linear_model import Lasso

# import function for elastic net regression
from sklearn.linear_model import ElasticNet

# import function to perform GridSearchCV
from sklearn.model_selection import GridSearchCV

In [ ]:
# load the csv file
df_lifeExp = pd.read_csv('Median_Prices.xls')

# display the first five observations
df_lifeExp.head()

In [ ]:
df_lifeExp.info()

# Data Prepration


In [ ]:
df_lifeExp.isnull().sum()

# Handling the missing values

In [ ]:
df_lifeExp.shape

In [ ]:
# display the total number of null values in each column of dataframe

# 'ascending = False' sorts values in the descending order
Total = df_lifeExp.isnull().sum().sort_values(ascending=False)          

# percentage of missing values
Percent = (df_lifeExp.isnull().sum()*100/df_lifeExp.isnull().count()).sort_values(ascending=False)  

# concat the 'Total' and 'Percent' columns using 'concat' function
# 'keys' is the list of column names
# 'axis = 1' concats along the columns
missing_data = pd.concat([Total, Percent], axis=1, keys=['Total', 'Percent'])    
missing_data

# We impute missing values

In [ ]:
df_missing_values = df_lifeExp[['CRIM','ZN','INDUS','RIVER','AGE','LSTAT']]

# plot histogram of all variables which have missing values
# set the number of bins to 20
# set the figure size using 'figsize'
df_missing_values.hist(bins = 20, figsize = (15,8))

# adjust the subplots
plt.tight_layout()

# display the plot
plt.show()

In [ ]:
df1 = ['CRIM','ZN','INDUS','AGE']
df_lifeExp[df1] = df_lifeExp[df1].fillna(df_lifeExp[df1].median())

In [ ]:
df2 = ['LSTAT']
df_lifeExp[df2] = df_lifeExp[df2].fillna(df_lifeExp[df2].mean())

In [ ]:
df_lifeExp.head()

In [ ]:
df_lifeExp.isnull().sum()

In [ ]:
df_lifeExp['RIVER'] = pd.get_dummies(data = df_lifeExp['RIVER'], drop_first=True)

In [ ]:
df_lifeExp.info()

In [ ]:
df_lifeExp['RIVER'] = df_lifeExp['RIVER'].fillna(df_lifeExp['RIVER'].mode())

In [ ]:
df_lifeExp.isnull().sum()

In [ ]:
df_lifeExp['RIVER'] = df_lifeExp['RIVER'].astype('float')
df_lifeExp.info()

In [ ]:
df_lifeExp.describe()

# Linear Regression


In [ ]:
# select the independent variables
X = df_lifeExp.iloc[:,:12]

# to estimate the regression coefficient , a constant term of '1' needs to be added as a separate column
# 'sm.add_constant' adds the intercept to the model
X = sm.add_constant(X)
X=X.astype("float")
# set the dependent variable
y = df_lifeExp['PRICE']

# split data into train subset and test subset
# set 'random_state' to generate the same dataset each time you run the code 
# 'test_size' returns the proportion of data to be included in the test set
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=10, test_size = 0.2)

# build a model with an intercept
MLR_model = sm.OLS(y_train, X_train).fit()
print(MLR_model.summary())

## Interpretation
- The multiple linear regression model is statistically significant and explains approximately 73.2% variation in house prices. Most explanatory variables significantly affect PRICE, especially RM (positive effect) and NOX/LSTAT (negative effects).
The model has good predictive ability and no major autocorrelation issue. However, the presence of multicollinearity and non-normal residuals suggests that the model can be further improved for better reliability and interpretation.


In [ ]:
# Level of Errors for full model (using OLS Estiamtion)
# predict the values of target variable using train data
y_pred_train = MLR_model.predict(X_train)
mse_train = mean_squared_error(y_train, y_pred_train)
print ("MSE for Train Data:",mse_train)
y_pred_test=MLR_model.predict(X_test)
mse_test = mean_squared_error(y_test, y_pred_test)
print ("MSE for Test Data:",mse_test)

## Interpretation
OLS Train MSE = 20.62 | Test MSE = 36.17
The test error is notably higher than the train error, suggesting the model has memorized some training-specific patterns.
However, the gap is not extreme, so overfitting is mild.
Presence of multicollinearity (detected later in heatmap) can inflate coefficient estimates, causing instability.


In [ ]:
#Cross Validation:
from sklearn.model_selection import KFold, cross_val_score

# 10-Fold Cross Validation
#Method 1: Using cross_val_score
model=LinearRegression()
kf = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=kf, scoring="r2")

print("R² scores for each fold:", scores)
print("Average R² score:", np.mean(scores))

## Interpretation
Average R² ≈ 0.704 across 10 folds — good explanatory power.
Fold scores are fairly consistent (no drastic swings), indicating the model is stable.
Conclusion: Some overfitting exists in the single split, but cross-validation confirms the model generalizes reasonably well.


# Gradient Descent

In [ ]:
# selecting the independent variables
X = df_lifeExp.iloc[:,:12]

# set the dependent variable
y = df_lifeExp['PRICE']

# initializing the standard scalar
X_scaler = StandardScaler()

# standardizing all the columns of df
X = X_scaler.fit_transform(X)

# standardizing the target explicitly and storing it in a new variable 'y'
y = (y - y.mean()) / y.std()

# split data into train subset and test subset
# set 'random_state' to generate the same dataset each time you run the code 
# 'test_size' returns the proportion of data to be included in the testing set
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=10, test_size = 0.2)

# hyper paramter tuning for full batcg


In [ ]:
# Full model using Full batch Gradient Descent (Hyper parameter tunning):
# Instantiate the SGDRegressor
sgd = SGDRegressor(max_iter=1000, tol=1e-3, penalty=None,learning_rate='constant', random_state=42)
#Define the parameter grid, eta0 is the learning rate in Learning rate in scikit-learn implementation
#param_grid ={'eta0': np.linspace(0.001, 1.0,10)}
param_grid = {'eta0': [1e-4, 1e-3, 1e-2, 1e-1, 1.0]}
#Grid Search
grid_search = GridSearchCV(estimator=sgd,param_grid=param_grid,scoring='neg_mean_squared_error',cv=5,n_jobs=-1)
#Fit grid search
grid_search.fit(X_train, y_train)
#Show best params
print("Best parameters found:", grid_search.best_params_)
# Evaluate on test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("\nTest set evaluation:")
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))

In [ ]:
# Full model using Full batch Gradient Descent:

# instantiate the SGDRegressor
# learning_rate='constant': fixed learning rate earning rate = eta0 at every step
# learning_rate='invscaling': decreases learning rate η (starting with eta0) over time
# learning_rate='adaptive' : decreases η (adaptive use of eta0)only when loss stops improving
# learning_rate='optimal' :automatically selects η (eta0 ignored) based on regularization and iteration
# set 'random_state' to generate the same dataset each time you run the code 
sgd = SGDRegressor(learning_rate='constant', eta0 = 0.001, random_state = 1)

# build the model on train data 
# use fit() to fit the model
linreg_with_SGD = sgd.fit(X_train, y_train)
# Make predictions on your train data
y_pred_train = linreg_with_SGD.predict(X_train)
# Make predictions on your test data
y_pred_test = linreg_with_SGD.predict(X_test)
mse_train = mean_squared_error(y_train, y_pred_train)
print ("MSE for Train Data:",mse_train)
mse_test = mean_squared_error(y_test, y_pred_test)
print ("MSE for Test Data:",mse_test)

## Interpretation
After tuning the learning rate using Grid Search, the full-batch gradient descent model achieved 
a training MSE of 0.250 and a testing MSE of 0.4126. Since the testing error is only slightly
greater than the training error, the model does not show significant overfitting and performs
well on unseen data.

In [ ]:
print("Intercept:", linreg_with_SGD.intercept_)
print("Coefficients:", linreg_with_SGD.coef_)

The intercept represents the predicted value of PRICE when all independent variables are zero. The coefficients represent the change in PRICE for a one-unit increase in the corresponding feature, keeping all other features constant.

Positive coefficient -> PRICE increases as that feature increases.
Negative coefficient -> PRICE decreases as that feature increases.

The fitted regression equation is given above.

## Examining the correlation among features in the linear regression model

In [ ]:
# set dimensions for the plot figure
fig_dims = (15,10)
fig, ax = plt.subplots(figsize=fig_dims)

# create a copy of dataframe
df_features = df_lifeExp.iloc[:,:12]

# plot the heat map
# corr: give the correlation matrix
# annot: prints the correlation values in the chart
# annot_kws: sets the font size of the annotation
sns.heatmap(df_features.corr(), annot = True, annot_kws = {"size": 10})

# rotate the y-axis labels
# set text size using 'fontsize'
plt.yticks(rotation = 'horizontal', fontsize = 15)
plt.xticks(fontsize = 15)

# display the plot
plt.show()

## Interpretation

# Regularisation

In [ ]:
# selecting the independent variables 
X =df_lifeExp.iloc[:,:12]

# set the dependent variable
y = df_lifeExp['PRICE']

# initializing the standard scalar
X_scaler = StandardScaler()

# standardizing all the columns of df_ipl
X = X_scaler.fit_transform(X)

# standardizing the target explicitly and storing it in a new variable 'y'
y = (y - y.mean()) / y.std()

# split data into train subset and test subset
# set 'random_state' to generate the same dataset each time you run the code 
# 'test_size' returns the proportion of data to be included in the testing set
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=110, test_size = 0.2)

# Ridge Regression

## Hyperparameter tuning for ridge

In [ ]:
# 'alpha' assigns the regularization strength to the model
tuned_paramaters = [{'alpha':np.linspace(0.1,100,1000)}]
 
# instantiate the Ridge() method
ridge = Ridge()

# use GridSearchCV() to find the optimal value of alpha
# estimator: pass the ridge regression model
# param_grid: pass the list 'tuned_parameters'
# cv: number of folds in k-fold i.e. here cv = 5
ridge_grid = GridSearchCV(estimator = ridge, 
                          param_grid = tuned_paramaters, 
                          cv = 10)

# fit the model on X_train and y_train using fit()
ridge_grid.fit(X_train, y_train)

# get the best parameters
print('Best parameters for Ridge Regression:', ridge_grid.best_params_)

In [ ]:
# use Ridge() to perform ridge regression
# 'alpha' assigns the regularization strength to the model
ridge = Ridge(alpha = 15.8)

# fit the model on train set using fit()
ridge_model = ridge.fit(X_train, y_train)

# For training set:
# train_pred: prediction made by the model on the training dataset 'X_train'
# y_train: actual values ofthe target variable for the train dataset
 
# predict the output of the target variable from the train data 
train_pred = ridge_model.predict(X_train)

# calculate the MSE using the "mean_squared_error" function
# MSE for the train data
mse_train = mean_squared_error(y_train, train_pred)

# print the MSE for the train set
print("MSE for Train Data: ", mse_train)
    
# For testing set:
# test_pred: prediction made by the model on the testing dataset 'y_train'
# y_test: actual values of the target variable for the test dataset
 
# predict the output of the target variable from the test data
test_pred = ridge_model.predict(X_test)

# MSE for the test data
mse_test = mean_squared_error(y_test, test_pred)

# print the MSE for the test set
print("MSE for Test Data:", mse_test)


## Interpretation
After Grid Search — Train MSE = 0.29 | Test MSE = 0.18
The slight increase in train MSE is expected: higher alpha shrinks coefficients more, making the model less flexible.
Interpretation of alpha=19: Moderate-to-strong regularization was optimal for this dataset.


In [ ]:
ridge_model.coef_

# LASSO

In [ ]:
# 'alpha' assigns the regularization strength to the model
tuned_paramaters = [{'alpha':np.linspace(0.1,100,1000)}]
 
# instantiate the Lasso() method
lasso = Lasso()

# use GridSearchCV() to find the optimal value of alpha
# estimator: pass the lasso regression model
# param_grid: pass the list 'tuned_parameters'
# cv: number of folds in k-fold i.e. here cv = 10
lasso_grid = GridSearchCV(estimator = lasso, 
                          param_grid = tuned_paramaters, 
                          cv = 10)

# fit the model on X_train and y_train using fit()
lasso_grid.fit(X_train, y_train)

# get the best parameters
print('Best parameters for Lasso Regression:', lasso_grid.best_params_)

In [ ]:
# use Lasso() to perform lasso regression
# 'alpha' 0.1 used as this was the best parameter value
lasso = Lasso(alpha = 0.1)

# fit the model on train set using fit()
lasso_model = lasso.fit(X_train, y_train)

# For training set:
# train_pred: prediction made by the model on the training dataset 'X_train'
# y_train: actual values ofthe target variable for the train dataset
 
# predict the output of the target variable from the train data 
train_pred = lasso_model.predict(X_train)

# calculate the MSE using the "mean_squared_error" function
# MSE for the train data
mse_train = mean_squared_error(y_train, train_pred)

# print the MSE for the train set
print("MSE for Train Data:", mse_train)
    
# For testing set:
# test_pred: prediction made by the model on the testing dataset 'y_train'
# y_test: actual values of the target variable for the test dataset
 
# predict the output of the target variable from the test data
test_pred = lasso_model.predict(X_test)

# MSE for the test data
mse_test = mean_squared_error(y_test, test_pred)

# print the MSE for the test set
print("MSE for Test Data:", mse_test)

## Interpretation
While a low Test MSE is generally a good thing, a Train MSE that is nearly double the Test MSE suggests that the model is likely 
underfitting the training data, and the evaluation metrics are heavily dependent on how the data was split.

In [ ]:
lasso_model.coef_

## Interpretation
The output lasso_model.coef_ reveals several coefficients that have been reduced to exactly 0. or -0.. 
This illustrates the built-in feature selection capability of Lasso Regression (L1 penalty), 
which effectively drops less important features by setting their weights to zero.

In [ ]:
# Grab the names of the first 12 columns directly from the original dataframe
feature_names = df_lifeExp.columns[:12]

# Pass the feature_names list to the 'Variable' column
df_lasso_coeff = pd.DataFrame(data={'Variable': feature_names, 'Coefficient': lasso_model.coef_})

# print the variables having the coefficient equal to zero
print('Insignificant variables obtained from Lasso Regression when alpha is 0.1:',
      df_lasso_coeff.Variable[df_lasso_coeff.Coefficient == 0].to_list())

# Elastic net

In [ ]:
# create a dictionary with hyperparameter values
tuned_paramaters = [{'alpha':[0.1,0.5,1,1.5,2,2.5,3,3.5,4],
                      'l1_ratio':[0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.85,0.9]}]
 
# instantiate the ElasticNet() method
enet = ElasticNet()

# using GridSearchCV to find the optimal value of alpha and l1_ratio
# estimator: pass the elastic net regression model
# param_grid: pass the list 'tuned_parameters'
# cv: number of folds in k-fold i.e. here cv = 10
enet_grid = GridSearchCV(estimator = enet, 
                          param_grid = tuned_paramaters, 
                          cv = 10)

# fit the model on X_train and y_train
enet_grid.fit(X_train, y_train)

# get the best parameters
print('Best parameters for ELastic-net Regression: ', enet_grid.best_params_, '\n')

In [ ]:
# use ElasticNet() to perform Elastic Net regression
# 'alpha' assigns the regularization strength to the model
# 'l1_ratio' is the ElasticNet mixing parameter
enet = ElasticNet(alpha = 0.1, l1_ratio = 0.5)

# fit the model on train data using fit()
enet_model = enet.fit(X_train, y_train)

# predict the values of target variable using test data
y_test_pred = enet_model.predict(X_test)

# For training set:
# train_pred: prediction made by the model on the training dataset 'X_train'
# y_train: actual values ofthe target variable for the train dataset
 
# predict the output of the target variable from the train data 
train_pred = enet_model.predict(X_train)

# calculate the MSE using the "mean_squared_error" function
# MSE for the train data
mse_train = mean_squared_error(y_train, train_pred)

# print the MSE for the train set
print("MSE for Train Data:", mse_train)
    
# For testing set:
# test_pred: prediction made by the model on the testing dataset 'y_train'
# y_test: actual values of the target variable for the test dataset
 
# predict the output of the target variable from the test data
test_pred = enet_model.predict(X_test)

# MSE for the test data
mse_test = mean_squared_error(y_test, test_pred)

# print the MSE for the test set
print("MSE for Test Data:", mse_test)

## Interpretation
Tuned Elastic Net: Train MSE = 0.3107 | Test MSE = 0.1867
l1_ratio=0.1 is close to 0 — the model behaves mostly like Ridge with minimal LASSO shrinkage.
AGE coefficient ≈ 0 — this variable has negligible independent effect on price.
Elastic Net achieved the lowest Test MSE 0.19 — best overall model


In [ ]:
enet_model.coef_

In [ ]:
# Grab the names of the first 12 columns directly from the original dataframe
feature_names = df_lifeExp.columns[:12]

# Pass the feature_names list to the 'Variable' column
df_enet_coeff = pd.DataFrame(data={'Variable': feature_names, 'Coefficient': enet_model.coef_})

# print the variables having the coefficient equal to zero
print('Insignificant variables obtained from Lasso Regression when alpha is 0.1:',
      df_enet_coeff.Variable[df_enet_coeff.Coefficient == 0].to_list())
